# NeuroGolf 2026 — Task 111 Deep Dive NeuroGolf Champions + Best Public Score

This project presents a highly optimized solution for **Task 111** in the NeuroGolf 2026 benchmark. The objective is to design an extremely compact and efficient ONNX model capable of perfectly solving the ARC-AGI transformation task while maximizing the competition scoring metric. The challenge emphasizes algorithmic reasoning, model compression, and memory-efficient deployment rather than large-scale neural network capacity.

Task 111 is structured around a deterministic visual reasoning problem. Each input grid contains a 10×10 active content area located in the upper-left region of the board. Within this content area, exactly one gray marker pixel (color value 5) serves as a reference point that determines the location of the target extraction region.

The solution first scans the input grid to identify the coordinates of the gray marker. Once the marker position is located, the model computes a fixed spatial offset relative to that pixel. Using the detected coordinates, it extracts a 3×3 window defined by:

* Rows: `row_gray + 1` through `row_gray + 3`
* Columns: `col_gray - 1` through `col_gray + 1`

This extracted region represents the complete semantic answer required by the task.

After extraction, the resulting 3×3 patch is placed into the upper-left corner of a new 30×30 output canvas. The remaining cells are filled with background values, producing a standardized output representation that satisfies the NeuroGolf evaluation format.

The entire pipeline is intentionally designed to avoid unnecessary computation. Instead of relying on deep convolutional stacks or transformer-based architectures, the model performs direct spatial reasoning through a minimal sequence of operations. This dramatically reduces memory consumption, parameter count, and inference latency while maintaining perfect task accuracy.

Key characteristics of the solution include:

* Precise detection of the unique gray anchor pixel.
* Deterministic coordinate transformation logic.
* Lightweight crop-and-place reasoning pipeline.
* Fixed-size 30×30 output generation.
* ONNX-compatible deployment.
* Near-minimal parameter footprint.
* Ultra-low memory overhead.
* Fast CPU inference suitable for constrained environments.

The optimization strategy focuses on maximizing NeuroGolf points under the competition scoring rule:

[
\text{Points} = \max(1.0,;25.0 - \log(\text{Mem_bytes} + \text{Params}))
]

where lower memory usage and fewer parameters directly increase the final score. Because the task can be represented through simple geometric reasoning, the solution achieves exceptional efficiency without sacrificing correctness.

### Performance Summary

| Metric          | Value                           |
| --------------- | ------------------------------- |
| Task ID         | 111                             |
| Framework       | ONNX                            |
| Memory Usage    | 772 bytes                       |
| Parameters      | 25                              |
| Output Size     | 30×30                           |
| Crop Size       | 3×3                             |
| Inference Style | Deterministic Spatial Reasoning |
| Public Score    | **18.319 Points**               |

### Why This Solution Performs Well

The task contains a strong structural prior: the gray pixel uniquely determines the answer location. By exploiting this property directly, the model avoids learning unnecessary representations and instead encodes the exact transformation rule. This allows the network to remain extremely small while still achieving perfect task behavior.

The resulting model demonstrates how symbolic spatial reasoning and careful engineering can outperform larger architectures in NeuroGolf-style compression challenges. It serves as a strong example of task-specific optimization, showing that understanding the underlying ARC-AGI pattern can be more valuable than increasing model complexity.

With only **772 bytes of memory**, **25 parameters**, and a **public score of 18.319**, this implementation represents a highly competitive solution for Task 111 and highlights the effectiveness of minimalist reasoning systems in resource-constrained AI competitions.


In [ ]:
# Install missing dependencies
import sys
try:
    import onnxruntime as ort
except ImportError:
    !{sys.executable} -m pip install onnxruntime
    import onnxruntime as ort

## 1. ARC Color Palette

In [ ]:
import json, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

arc_colors = [
    '#000000',  # 0: black
    '#0074D9',  # 1: blue
    '#FF4136',  # 2: red
    '#2ECC40',  # 3: green
    '#FFDC00',  # 4: yellow
    '#AAAAAA',  # 5: gray
    '#F012BE',  # 6: magenta
    '#FF851B',  # 7: orange
    '#7FDBCA',  # 8: teal
    '#870C25',  # 9: maroon
]
color_names = ['black','blue','red','green','yellow','gray','magenta','orange','teal','maroon']

fig, ax = plt.subplots(figsize=(12, 2))
for i, (c, nm) in enumerate(zip(arc_colors, color_names)):
    ax.add_patch(patches.Rectangle((i*1.2, 0), 1, 1, facecolor=c, edgecolor='gray', lw=1))
    ax.text(i*1.2+0.5, 0.5, str(i), ha='center', va='center', fontsize=16, fontweight='bold',
            color='white' if i in [0,9] else 'black')
    ax.text(i*1.2+0.5, -0.2, nm, ha='center', va='top', fontsize=8)
ax.set_xlim(-0.5, 12.5)
ax.set_ylim(-0.3, 1.5)
ax.axis('off')
plt.show()

## 2. Task 111 Examples

The input is represented as a **30×30 one-hot encoded grid**, but all meaningful information is located inside the **top-left 10×10 content region**. The remaining cells act as padding and do not influence the solution.

The key feature of the task is a **single gray pixel (color 5)**. This gray pixel acts as an anchor point that determines where the model should extract the target pattern.

### Step 1 — Read the Input Grid

The model receives a 30×30 input grid.

```text
30×30 Grid
┌─────────────────────────────┐
│ 10×10 Content Region        │
│                             │
│ Remaining Area = Padding    │
└─────────────────────────────┘
```

Only the top-left 10×10 area contains relevant information.

---

### Step 2 — Locate the Gray Pixel

Search the content region for the unique gray pixel.

Example:

```text
0 0 0 0 0
0 0 0 0 0
0 0 5 0 0
0 0 0 0 0
0 0 0 0 0
```

The gray pixel is located at:

```text
(row_gray, col_gray) = (2, 2)
```

This coordinate becomes the reference point for the crop operation.

---

### Step 3 — Compute the Crop Window

Using the gray pixel location, define the crop boundaries:

```text
Rows    : row_gray + 1 → row_gray + 3
Columns : col_gray - 1 → col_gray + 1
```

In code form:

```python
crop = grid[
    row_gray + 1 : row_gray + 4,
    col_gray - 1 : col_gray + 2
]
```

This always produces a:

```text
3×3 region
```

---

### Step 4 — Extract the 3×3 Patch

Suppose the surrounding area looks like:

```text
0 0 0 0 0
0 0 5 0 0
1 2 3 4 5
6 7 8 9 1
2 3 4 5 6
```

The crop window begins one row below the gray pixel and spans three rows and three columns.

Extracted patch:

```text
1 2 3
6 7 8
2 3 4
```

This 3×3 patch is the answer pattern.

---

### Step 5 — Create an Empty 30×30 Output Grid

Initialize a new output canvas:

```text
30×30
```

filled with background values.

```python
output = zeros((30, 30))
```

---

### Step 6 — Place the Patch in the Upper-Left Corner

Copy the extracted patch into the top-left corner of the output.

```python
output[0:3, 0:3] = crop
```

Result:

```text
1 2 3 . . . . .
6 7 8 . . . . .
2 3 4 . . . . .
. . . . . . . .
. . . . . . . .
```

The dots represent background cells.

---

### Step 7 — Return the Final Output

The final prediction is a 30×30 grid containing:

* The extracted 3×3 patch at the upper-left corner.
* Background values everywhere else.
* No additional transformations, rotations, or color changes.

```text
Input
   ↓
Find Gray Pixel (5)
   ↓
Compute Relative Crop Window
   ↓
Extract 3×3 Patch
   ↓
Create Empty 30×30 Output
   ↓
Paste Patch at (0,0)
   ↓
Return Output
```

This deterministic pipeline completely solves Task 111 while requiring only minimal memory and parameter usage, making it ideal for high-scoring NeuroGolf submissions.


In [ ]:
# ── Inline task data (4 ARC-AGI examples) ──
EXAMPLE_INPUTS = [
  [[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,1,1],[0,0,0,5,0,0,0,1,1,0],[0,0,0,1,0,0,0,0,1,0],[0,0,1,1,1,0,0,0,0,0],[0,0,0,1,1,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,1,1,0,0],[0,0,0,0,0,1,1,1,0,0],[0,0,0,0,0,0,1,1,0,0]],
  [[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,5,0,0],[0,0,0,0,0,0,4,4,0,0],[0,0,4,0,0,0,0,0,4,0],[0,4,0,4,0,0,0,4,0,0],[0,0,4,4,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0]],
  [[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0],[0,0,2,2,0,0,0,0,0,0],[0,2,0,2,0,0,0,0,0,0],[0,0,2,0,0,0,0,5,0,0],[0,0,0,0,0,0,0,2,2,0],[0,0,0,0,0,0,2,2,0,0],[0,0,0,0,0,0,0,2,0,0],[0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0]],
  [[0,0,0,0,0,0,5,0,0,0],[0,0,0,0,0,0,3,0,0,0],[0,0,0,0,0,3,3,0,0,0],[0,0,0,0,0,0,3,3,0,0],[0,0,3,0,0,0,0,0,0,0],[0,3,3,0,0,0,0,0,0,0],[0,0,3,0,0,0,3,0,0,0],[0,0,0,0,0,3,3,3,0,0],[0,0,0,0,0,0,3,3,0,0],[0,0,0,0,0,0,0,0,0,0]],
]
EXAMPLE_OUTPUTS = [
  [[0,1,0],[1,1,1],[0,1,1]],
  [[4,4,0],[0,0,4],[0,4,0]],
  [[0,2,2],[2,2,0],[0,2,0]],
  [[0,3,0],[3,3,0],[0,3,3]],
]
EXAMPLE_LABELS = ['Train 1','Train 2','Train 3','Test 1']
GRAY_POSITIONS = [(2,3),(1,7),(4,7),(0,6)]
CROP_FORMULAS = ['[3:6, 2:5]','[2:5, 6:9]','[5:8, 6:9]','[1:4, 5:8]']

def plot_arc_grid(grid, ax, title=''):
    H, W = len(grid), len(grid[0])
    img = np.zeros((H, W, 3), dtype=np.uint8)
    for r in range(H):
        for c in range(W):
            hex_c = arc_colors[grid[r][c]].lstrip('#')
            img[r,c] = [int(hex_c[i:i+2], 16) for i in (0, 2, 4)]
    ax.imshow(img, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title, fontsize=10, fontweight='bold')
    for r in range(H+1): ax.axhline(r-0.5, color='gray', lw=0.5, alpha=0.3)
    for c in range(W+1): ax.axvline(c-0.5, color='gray', lw=0.5, alpha=0.3)

fig, axes = plt.subplots(4, 3, figsize=(9, 10))
for i in range(4):
    label = EXAMPLE_LABELS[i]
    r0, c0 = GRAY_POSITIONS[i]
    inp = EXAMPLE_INPUTS[i]
    out = EXAMPLE_OUTPUTS[i]
    plot_arc_grid(inp, axes[i][0], f'{label}: Input (10×10)')
    # Highlight gray pixel
    axes[i][0].plot(c0, r0, marker='*', markersize=10, color='red', markeredgecolor='white', markeredgewidth=1)
    axes[i][0].text(c0-0.4, r0-0.8, 'gray(5)', fontsize=7, color='red', fontweight='bold')
    plot_arc_grid(out, axes[i][1], f'{label}: Output (3×3)')
    # Show crop region on input
    crop_desc = CROP_FORMULAS[i]
    axes[i][0].add_patch(patches.Rectangle((c0-1-0.5, r0+1-0.5), 3, 3,
        linewidth=2, edgecolor='lime', facecolor='none', ls='--'))
    axes[i][0].text(c0-0.5, r0+2.5, crop_desc, fontsize=7, color='lime', fontweight='bold')
    # Explanation
    axes[i][2].axis('off')
    axes[i][2].text(0, 0.7, f'Gray at ({r0},{c0})', fontsize=10, fontweight='bold')
    axes[i][2].text(0, 0.5, f'Crop rows [{r0+1}:{r0+4})', fontsize=9)
    axes[i][2].text(0, 0.35, f'Crop cols [{c0-1}:{c0+2})', fontsize=9)
    axes[i][2].text(0, 0.15, f'{crop_desc}', fontsize=9)
plt.tight_layout()
plt.show()

## 3. ONNX Model Architecture

The model is a single `[1,10,30,30]` → `[1,10,30,30]` ONNX graph.
Here is the **exact source code** (89 lines) — each node is explained below.

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper

OPSET = 11
i = helper.make_tensor_value_info("input", 1, [1, 10, 30, 30])
o = helper.make_tensor_value_info("output", 1, [1, 10, 30, 30])

const_1 = numpy_helper.from_array(np.array([1], dtype=np.int64), "const_1")
const_2 = numpy_helper.from_array(np.array([2], dtype=np.int64), "const_2")
const_3 = numpy_helper.from_array(np.array([3], dtype=np.int64), "const_3")
pads_27 = numpy_helper.from_array(np.array([0,0,0,0,0,0,27,27], dtype=np.int64), "pads_27")
axes_0123 = numpy_helper.from_array(np.array([0,1,2,3], dtype=np.int64), "axes_0123")
axes_23 = numpy_helper.from_array(np.array([2,3], dtype=np.int64), "axes_23")
start_gray = numpy_helper.from_array(np.array([0,5,0,0], dtype=np.int64), "start_gray")
end_gray = numpy_helper.from_array(np.array([1,6,7,9], dtype=np.int64), "end_gray")

nodes = [
    # 1. Extract gray channel from 7×9 content region
    helper.make_node("Slice", ["input","start_gray","end_gray","axes_0123"], ["gray_f32"]),
    # 2. Row projection via ReduceSum over columns
    helper.make_node("ReduceSum", ["gray_f32"], ["row_proj"], axes=[3], keepdims=1),
    helper.make_node("ArgMax", ["row_proj"], ["n"], axis=2, keepdims=0),
    # 3. Column projection via ReduceSum over rows
    helper.make_node("ReduceSum", ["gray_f32"], ["col_proj"], axes=[2], keepdims=1),
    helper.make_node("ArgMax", ["col_proj"], ["m"], axis=3, keepdims=0),
    # 4. Flatten n,m to 1D
    helper.make_node("Reshape", ["n", "const_1"], ["n_1d"]),
    helper.make_node("Reshape", ["m", "const_1"], ["m_1d"]),
    # 5. Compute crop bounds: row in [n+1, n+4), col in [m-1, m+2)
    helper.make_node("Add", ["n_1d", "const_1"], ["row_start"]),
    helper.make_node("Add", ["row_start", "const_3"], ["row_end"]),
    helper.make_node("Sub", ["m_1d", "const_1"], ["col_start"]),
    helper.make_node("Add", ["m_1d", "const_2"], ["col_end"]),
    # 6. Concat row/col into 2-element slice params
    helper.make_node("Concat", ["row_start","col_start"], ["starts_rc"], axis=0),
    helper.make_node("Concat", ["row_end","col_end"], ["ends_rc"], axis=0),
    # 7. Crop from full input (axes [2,3] = row, col only)
    helper.make_node("Slice", ["input","starts_rc","ends_rc","axes_23"], ["crop_f32"]),
    # 8. Pad to 30×30
    helper.make_node("Pad", ["crop_f32", "pads_27"], ["output"], mode="constant"),
]

inits = [const_1, const_2, const_3, pads_27, axes_0123, axes_23, start_gray, end_gray]
graph = helper.make_graph(nodes, "g", [i], [o], initializer=inits)
model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", OPSET)])
model.ir_version = 8

### Node-by-Node Walkthrough

| Step | Node | Input → Output | What it does |
|------|------|----------------|-------------|
| **1** | `Slice` | `input[0:1, 5:6, 0:7, 0:9]` → `gray_f32 [1,1,7,9]` | Extract channel 5 (gray). The 30×30 input is one-hot F32; we keep only the gray channel. End row=7 (not 10) saves memory — gray pixel is always in rows 0–6, cols 0–8. |
| **2a** | `ReduceSum` | `gray_f32` along axis=3 → `row_proj [1,1,7,1]` | Sum columns → each row's total. Since there's exactly 1 gray pixel, one entry = 1, rest = 0. |
| **2b** | `ArgMax` | `row_proj` axis=2 → `n [1,1,1]` | Finds the row index `n` where the gray pixel sits. |
| **3a** | `ReduceSum` | `gray_f32` along axis=2 → `col_proj [1,1,1,9]` | Sum rows → each column's total. |
| **3b** | `ArgMax` | `col_proj` axis=3 → `m [1,1,1]` | Finds the column index `m`. |
| **4** | `Reshape` ×2 | `n` → `n_1d [1]`, `m` → `m_1d [1]` | Flatten 3D ArgMax output to 1D for arithmetic. |
| **5** | `Add` ×3, `Sub` | `n_1d + 1` → `row_start`; `row_start + 3` → `row_end`; `m_1d - 1` → `col_start`; `m_1d + 2` → `col_end` | Compute the half-open crop region: rows `[n+1, n+4)`, cols `[m-1, m+2)`. |
| **6** | `Concat` ×2 | `[row_start, col_start]` → `starts_rc [2]`; `[row_end, col_end]` → `ends_rc [2]` | Build 2-element slice params (just row,col dims). |
| **7** | `Slice` | `input[..., starts_rc:ends_rc, axes=[2,3]]` → `crop_f32 [1,10,3,3]` | Crop the 3×3 region from the full one-hot input, keeping all 10 channels. |
| **8** | `Pad` | `crop_f32` + `[0,0,0,0,0,0,27,27]` → `output [1,10,30,30]` | Place the 3×3 crop at top-left; pad right/bottom with zeros. **Free!** The output tensor is excluded from memory scoring. |

## 4. Cost Breakdown

The scoring formula: `Score = 25 - log(Mem + Params)`

In [ ]:
# Memory by tensor
tensors = [
    ('gray_f32',  'F32', [1,1,7,9],       252),
    ('row_proj',  'F32', [1,1,7,1],        28),
    ('col_proj',  'F32', [1,1,1,9],        36),
    ('n',         'I64', [1,1,1],           8),
    ('m',         'I64', [1,1,1],           8),
    ('n_1d,m_1d', 'I64', [1],              16),
    ('row_start..col_end', 'I64', [1] * 4, 32),
    ('starts_rc', 'I64', [2],              16),
    ('ends_rc',   'I64', [2],              16),
    ('crop_f32',  'F32', [1,10,3,3],      360),
    ('output',    'F32', [1,10,30,30],      0),  # excluded
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
names = [t[0] for t in tensors[:-1]]
mems  = [t[3] for t in tensors[:-1]]
colors_pie = ['#e74c3c','#e67e22','#f39c12','#9b59b6','#9b59b6','#3498db',
              '#1abc9c','#2ecc71','#2ecc71','#c0392b']
wedges, texts, autotexts = ax.pie(mems, labels=names, autopct='%1.0f%%',
    colors=colors_pie, startangle=90, textprops={'fontsize': 8})
for t in autotexts: t.set_fontsize(7)
ax.set_title('Memory Distribution (772 B total)', fontsize=11, fontweight='bold')

ax = axes[1]
ax.axis('off')
total_mem = sum(t[3] for t in tensors)
total_params = 25
total_cost = total_mem + total_params
score = max(1.0, 25.0 - math.log(max(1.0, total_cost)))

stats_text = (
    f"Memory (sum of intermediates):   {total_mem:>6,} B\n"
    f"Parameters (initializer elements): {total_params:>6}\n"
    f"Total cost (Mem + Params):       {total_cost:>6,}\n"
    f"Score = 25 - log({total_cost}) = {score:.3f} pts\n"
)
ax.text(0.1, 0.6, stats_text, fontsize=12, fontfamily='monospace', verticalalignment='top')

# Initializers breakdown
init_text = (
    "Initializers (params cost):\n"
    "  const_1, const_2, const_3:    1 each → 3\n"
    "  pads_27 [0,0,0,0,0,0,27,27]: 8\n"
    "  axes_0123 [0,1,2,3]:         4\n"
    "  axes_23 [2,3]:               2\n"
    "  start_gray [0,5,0,0]:        4\n"
    "  end_gray [1,6,7,9]:          4\n"
    "  ─────────────────────────────────\n"
    "  Total:                      25 params"
)
ax.text(0.1, 0.05, init_text, fontsize=9, fontfamily='monospace', verticalalignment='bottom')

plt.tight_layout()
plt.show()

## 5. Running the Model

We can use ONNX Runtime to verify the model on the examples above.

In [ ]:
import onnx
from onnx import helper, numpy_helper

def convert_grid_to_input(grid):
    """Convert ARC grid [H,W] to one-hot [1,10,30,30]."""
    batch = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r in range(len(grid)):
        for c in range(len(grid[r])):
            if grid[r][c] != 0:
                batch[0, grid[r][c], r, c] = 1.0
    return batch

def convert_output_to_grid(output):
    """Convert one-hot [1,10,30,30] to ARC grid [H,W]."""
    grid = np.zeros((30, 30), dtype=np.int64)
    for r in range(30):
        for c in range(30):
            for ch in range(10):
                if output[0, ch, r, c] > 0:
                    grid[r, c] = ch
                    break
    return grid

# Build the ONNX model in-memory
const_1 = numpy_helper.from_array(np.array([1], dtype=np.int64), "const_1")
const_2 = numpy_helper.from_array(np.array([2], dtype=np.int64), "const_2")
const_3 = numpy_helper.from_array(np.array([3], dtype=np.int64), "const_3")
pads_27 = numpy_helper.from_array(np.array([0,0,0,0,0,0,27,27], dtype=np.int64), "pads_27")
axes_0123 = numpy_helper.from_array(np.array([0,1,2,3], dtype=np.int64), "axes_0123")
axes_23 = numpy_helper.from_array(np.array([2,3], dtype=np.int64), "axes_23")
start_gray = numpy_helper.from_array(np.array([0,5,0,0], dtype=np.int64), "start_gray")
end_gray = numpy_helper.from_array(np.array([1,6,7,9], dtype=np.int64), "end_gray")

nodes = [
    helper.make_node("Slice", ["input","start_gray","end_gray","axes_0123"], ["gray_f32"]),
    helper.make_node("ReduceSum", ["gray_f32"], ["row_proj"], axes=[3], keepdims=1),
    helper.make_node("ArgMax", ["row_proj"], ["n"], axis=2, keepdims=0),
    helper.make_node("ReduceSum", ["gray_f32"], ["col_proj"], axes=[2], keepdims=1),
    helper.make_node("ArgMax", ["col_proj"], ["m"], axis=3, keepdims=0),
    helper.make_node("Reshape", ["n", "const_1"], ["n_1d"]),
    helper.make_node("Reshape", ["m", "const_1"], ["m_1d"]),
    helper.make_node("Add", ["n_1d", "const_1"], ["row_start"]),
    helper.make_node("Add", ["row_start", "const_3"], ["row_end"]),
    helper.make_node("Sub", ["m_1d", "const_1"], ["col_start"]),
    helper.make_node("Add", ["m_1d", "const_2"], ["col_end"]),
    helper.make_node("Concat", ["row_start","col_start"], ["starts_rc"], axis=0),
    helper.make_node("Concat", ["row_end","col_end"], ["ends_rc"], axis=0),
    helper.make_node("Slice", ["input","starts_rc","ends_rc","axes_23"], ["crop_f32"]),
    helper.make_node("Pad", ["crop_f32", "pads_27"], ["output"], mode="constant"),
]

i = helper.make_tensor_value_info("input", 1, [1,10,30,30])
o = helper.make_tensor_value_info("output", 1, [1,10,30,30])
graph = helper.make_graph(nodes, "g", [i], [o],
    initializer=[const_1, const_2, const_3, pads_27, axes_0123, axes_23, start_gray, end_gray])
model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", 11)])
model.ir_version = 8

# Run inference
import onnxruntime as ort
session = ort.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])

fig, axes = plt.subplots(4, 3, figsize=(9, 10))
all_ok = True
for i in range(4):
    label = EXAMPLE_LABELS[i]
    r0, c0 = GRAY_POSITIONS[i]
    inp = EXAMPLE_INPUTS[i]
    exp = EXAMPLE_OUTPUTS[i]
    model_in = convert_grid_to_input(inp)
    model_out = session.run(None, {'input': model_in})[0]
    pred = convert_output_to_grid(model_out)
    pred_crop = pred[:3, :3]
    
    plot_arc_grid(inp, axes[i][0], f'{label}: Input')
    axes[i][0].plot(c0, r0, marker='*', markersize=10, color='red', markeredgecolor='white', markeredgewidth=1)
    plot_arc_grid(exp, axes[i][1], f'{label}: Expected')
    plot_arc_grid(pred_crop, axes[i][2], f'{label}: Model Out')
    
    match = np.array_equal(np.array(exp), pred_crop)
    if not match:
        axes[i][2].set_title(f'{label}: ✗ MISMATCH', fontsize=10, fontweight='bold', color='red')
        all_ok = False
    else:
        axes[i][2].set_title(f'{label}: ✓ MATCH', fontsize=10, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

score = max(1.0, 25.0 - math.log(max(1.0, 772 + 25)))
print(f'All 4 examples correct: {all_ok}')
print(f'Score: Mem=772 + Params=25 = 797 → {score:.3f} pts')

## ![](https://arcprize.org/arc-agi/3)6. Summary

| Aspect | Detail |
|--------|--------|
| **Task** | Crop 3×3 region below-and-left of gray pixel in 10×10 content |
| **ONNX ops** | `Slice`, `ReduceSum`, `ArgMax`, `Reshape`, `Add`, `Sub`, `Concat`, `Pad` |
| **Memory** | 772 B (60% from gray extraction + 40% from crop) |
| **Params** | 25 (8 constant initializers for slice bounds, axes, padding) |
| **Score** | **18.319 points** |

### Key takeaways for other tasks:
- **Reduce early**: shrink tensors with ReduceSum/ReduceMax before any other operation
- **F32 is fine for small tensors**: Cast to F16 only saves memory when tensors are >200 B
- **Graph output is free**: Name your final node `"output"` to exclude it from scoring
- **Slice > Gather**: A targeted Slice avoids the full-channel intermediate (i.e. is like two gather)

# Update best public score
We thanks @anazemcev and @seddiktrk for his shared notebook with the currently best available solution, also thanks to all the others that share useful contents.

In [ ]:
import shutil
import os
import zipfile

# --- CONFIGURATION ---
SOURCE_FOLDER = '/kaggle/input/datasets/massimilianoghiotto/neurogolf2026-6110/submission'
OUTPUT_ZIP = '/kaggle/working/submission.zip'

# Package the ZIP (Ensuring files are at the root)
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(SOURCE_FOLDER):
        for file in files:
            if file.endswith('.onnx'):
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, SOURCE_FOLDER))